In [ ]:
import json
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer


In [ ]:
import kagglehub

path = kagglehub.dataset_download("Cornell-University/arxiv")
file_path = path + "/arxiv-metadata-oai-snapshot.json"

100%|██████████| 1.60G/1.60G [00:19<00:00, 88.3MB/s]

Extracting files...


In [ ]:
def get_class(categories_str):
    first = categories_str.split()[0]

    if first.startswith("cs."):
        return "cs"
    if first.startswith("econ."):
        return "econ"
    if first.startswith("eess."):
        return "eess"
    if first.startswith("math."):
        return "math"
    if first.startswith("physics."):
        return "physics"
    if first.startswith("q-bio."):
        return "q-bio"
    if first.startswith("q-fin."):
        return "q-fin"
    if first.startswith("stat."):
        return "stat"

    physics_subclasses = [
        "astro-ph", "cond-mat", "gr-qc", "hep-ex", "hep-lat",
        "hep-ph", "hep-th", "math-ph", "nlin", "nucl-ex",
        "nucl-th", "quant-ph"
    ]

    for p in physics_subclasses:
        if first.startswith(p):
            return "physics"

    return None

In [ ]:
target_per_class = 1600
target_classes = ["cs", "econ", "eess", "math", "physics", "q-bio", "q-fin", "stat"]

data = {c: [] for c in target_classes}

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        title = obj.get("title", "").strip()
        abstract = obj.get("abstract", "").strip()
        categories = obj.get("categories", "").strip()

        label = get_class(categories)

        if label is None:
            continue
        if label not in data:
            continue
        if not title and not abstract:
            continue

        if len(data[label]) < target_per_class:
            data[label].append({
                "title": title,
                "abstract": abstract,
                "label": label
            })

        if all(len(data[c]) >= target_per_class for c in target_classes):
            break

rows = []
for c in target_classes:
    rows.extend(data[c])

df = pd.DataFrame(rows)

print(df.shape)
# print(df["label"].value_counts())
df.head()

(12800, 3)


,title,abstract,label
0,Intelligent location of simultaneously active ...,The intelligent acoustic emission locator is d...,cs
1,Intelligent location of simultaneously active ...,Part I describes an intelligent acoustic emiss...,cs
2,On-line Viterbi Algorithm and Its Relationship...,"In this paper, we introduce the on-line Viterb...",cs
3,Real Options for Project Schedules (ROPS),Real Options for Project Schedules (ROPS) has ...,cs
4,Sparsely-spread CDMA - a statistical mechanics...,"Sparse Code Division Multiple Access (CDMA), a...",cs


In [ ]:
classes = sorted(df["label"].unique())
classes_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_classes = {i: c for i, c in enumerate(classes)}
df["label_id"] = df["label"].map(classes_to_idx)
df["text"] = "Title: " + df["title"].fillna("") + " Abstract: " + df["abstract"].fillna("")
print(classes, classes_to_idx, idx_to_classes)

['cs', 'econ', 'eess', 'math', 'physics', 'q-bio', 'q-fin', 'stat'] {'cs': 0, 'econ': 1, 'eess': 2, 'math': 3, 'physics': 4, 'q-bio': 5, 'q-fin': 6, 'stat': 7} {0: 'cs', 1: 'econ', 2: 'eess', 3: 'math', 4: 'physics', 5: 'q-bio', 6: 'q-fin', 7: 'stat'}


In [ ]:
train_df, other_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(other_df, test_size=0.5, random_state=42, stratify=other_df["label"])

In [ ]:
test_df[["title", "abstract"]].sample(50, random_state=42).to_csv("test_examples.csv", index=False)

In [ ]:
train_ds = Dataset.from_pandas(train_df[["text", "label_id"]])
val_ds = Dataset.from_pandas(val_df[["text", "label_id"]])
test_ds = Dataset.from_pandas(test_df[["text", "label_id"]])

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", max_length=256, truncation=True)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label_id", "labels")
val_ds = val_ds.rename_column("label_id", "labels")
test_ds = test_ds.rename_column("label_id", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Map:   0%|          | 0/1280 [00:00<?, ? examples/s]

Map:   0%|          | 0/1280 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(classes))

training_args = TrainingArguments(
    output_dir="./basic_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    logging_strategy="epoch",
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.705292,0.486589,0.841406,0.840721
2,0.369366,0.515693,0.854688,0.854725


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2560, training_loss=0.5373286128044128, metrics={'train_runtime': 518.692, 'train_samples_per_second': 39.484, 'train_steps_per_second': 4.935, 'total_flos': 1356611306127360.0, 'train_loss': 0.5373286128044128, 'epoch': 2.0})

In [ ]:
trainer.evaluate(val_ds)

{'eval_loss': 0.5156932473182678,
 'eval_accuracy': 0.8546875,
 'eval_f1_macro': 0.8547245568050832,
 'eval_runtime': 9.8031,
 'eval_samples_per_second': 130.571,
 'eval_steps_per_second': 16.321,
 'epoch': 2.0}

In [ ]:
train_sci = Dataset.from_pandas(train_df[["text", "label_id"]])
val_sci = Dataset.from_pandas(val_df[["text", "label_id"]])
test_sci = Dataset.from_pandas(test_df[["text", "label_id"]])

model2_name = "allenai/scibert_scivocab_uncased"
tokenizer2 = AutoTokenizer.from_pretrained(model2_name)

def tokenize_sci(batch):
    return tokenizer2(batch["text"], padding="max_length", max_length=256, truncation=True)

train_sci = train_sci.map(tokenize_sci, batched=True)
val_sci = val_sci.map(tokenize_sci, batched=True)
test_sci = test_sci.map(tokenize_sci, batched=True)

train_sci = train_sci.rename_column("label_id", "labels")
val_sci = val_sci.rename_column("label_id", "labels")
test_sci = test_sci.rename_column("label_id", "labels")

train_sci.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_sci.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_sci.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Map:   0%|          | 0/1280 [00:00<?, ? examples/s]

Map:   0%|          | 0/1280 [00:00<?, ? examples/s]

In [ ]:
model2 = AutoModelForSequenceClassification.from_pretrained(model2_name, num_labels=len(classes))

training_args = TrainingArguments(
    output_dir="./scibert_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    logging_strategy="epoch",
    save_total_limit=2,
)

trainer = Trainer(
    model=model2,
    args=training_args,
    train_dataset=train_sci,
    eval_dataset=val_sci,
    compute_metrics=compute_metrics
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.603081,0.517912,0.843750,0.841137
2,0.330610,0.505834,0.873437,0.873793
3,0.180948,0.560385,0.886719,0.887030


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3840, training_loss=0.37154595454533895, metrics={'train_runtime': 1566.5333, 'train_samples_per_second': 19.61, 'train_steps_per_second': 2.451, 'total_flos': 4041603526164480.0, 'train_loss': 0.37154595454533895, 'epoch': 3.0})

In [ ]:
trainer.evaluate(val_sci)

{'eval_loss': 0.5603849291801453,
 'eval_accuracy': 0.88671875,
 'eval_f1_macro': 0.8870297527599033,
 'eval_runtime': 18.8606,
 'eval_samples_per_second': 67.866,
 'eval_steps_per_second': 8.483,
 'epoch': 3.0}

In [ ]:
trainer.save_model("./final_model")
tokenizer2.save_pretrained("./final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [ ]:
!zip -r final_model.zip final_model

  adding: final_model/ (stored 0%)
  adding: final_model/model.safetensors (deflated 7%)
  adding: final_model/config.json (deflated 57%)
  adding: final_model/tokenizer_config.json (deflated 42%)
  adding: final_model/tokenizer.json (deflated 71%)
  adding: final_model/training_args.bin (deflated 53%)


In [ ]:
from google.colab import files
files.download("final_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>